In [1]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error 
from scipy import stats
import sys
import scipy.signal

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "/home/40265864@ecit.qub.ac.uk/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.adc.trig_count                     changed from 3992641757                to 3994592945               
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx         

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd firmware/XOR_test
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
.
Welcome to another exciting ChipWhisperer target build!!
+--------------------------------------------------------
Compiling:
+ Built for platform Microchip SAM4S with:
-en     XOR_test.c ...
+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



XOR_test.c: In function 'main':
XOR_test.c:144:13: warning: variable 'xor' set but not used [-Wunused-but-set-variable]
  144 |         int xor;
      |             ^~~


-e Done!
.
LINKING:
-en     XOR_test-CW308_SAM4S.elf ...
-e Done!
.
.
.
.
.
Creating load file for Flash: XOR_test-CW308_SAM4S.hex
Creating load file for Flash: XOR_test-CW308_SAM4S.bin
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature XOR_test-CW308_SAM4S.elf XOR_test-CW308_SAM4S.hex
Creating load file for EEPROM: XOR_test-CW308_SAM4S.eep
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature XOR_test-CW308_SAM4S.elf XOR_test-CW308_SAM4S.bin
Creating Extended Listing: XOR_test-CW308_SAM4S.lss
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex XOR_test-CW308_SAM4S.elf XOR_test-CW308_SAM4S.eep || exit 0
arm-none-eabi-objdump -h -S -z XOR_test-CW308_SAM4S.elf > XOR_test-CW308_SAM4S.lss
Creating Symbol Table: XOR_test-CW308_SAM4S.sym
arm-none-eabi-nm -n XOR_test-CW308_SAM4S.elf > XOR_test-CW308_SAM4S.sym
Size after:
   text	   data	    bss	    dec	    hex	filenam

In [5]:
cw.program_target(scope, prog, "firmware/XOR_test/XOR_test-{}.hex".format(PLATFORM))

In [5]:
scope.adc.samples = 2000                 # Number of samples per segment
scope.adc.stream_mode = "segmented"            # Enable segmented capture
scope.adc.segments = 1000   

In [6]:
def align_traces(traces):
    ref_trace = traces[0]  # Use the first trace as reference
    aligned_traces = []

    for trace in traces:
        correlation = np.correlate(trace, ref_trace, mode="full")  # Compute cross-correlation
        shift = np.argmax(correlation) - (len(trace) - 1)  # Find best alignment
        aligned_trace = np.roll(trace, -shift)  # Shift the trace
        aligned_traces.append(aligned_trace)
    
    return np.array(aligned_traces)

In [7]:
def get_trace():
    #num_char = target.in_waiting()
    #while num_char > 0:
        #target.read(num_char, 10)
        #time.sleep(0.01)
        #num_char = target.in_waiting()
    time.sleep(0.1)
    #target.flush()
    scope.arm()
    target.write("Start\n")
    if scope.capture():
        raise RuntimeError("Capture failed")
    trace_segments = scope.get_last_trace_segmented()
    alligned_traces = align_traces(trace_segments)
    return alligned_traces

In [8]:
reset_target(scope)

traces = get_trace()

In [11]:
cw.plot(traces[0]) * cw.plot(traces[1]) * cw.plot(traces[2]) * cw.plot(traces[3])

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [16]:
avg0 = []
avg1 = []
avg2 = []
avg3 = []
for i in range(250):
    avg0.append(traces[int(4*i)])
    avg1.append(traces[int(4*i +1)])    
    avg2.append(traces[int(4*i +2)])    
    avg3.append(traces[int(4*i +3)])
avg0 = np.mean(avg0, axis=0)
avg1 = np.mean(avg1, axis=0)
avg2 = np.mean(avg2, axis=0)
avg3 = np.mean(avg3, axis=0)

In [18]:
cw.plot(avg0) * cw.plot(avg1) * cw.plot(avg2) * cw.plot(avg3)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [25]:
cw.plot(avg2 - avg3)

:Curve   [x]   (y)